[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kunifujiwara/VoxCity/blob/main/docs/examples/demo_data_sources.ipynb) · part of the [VoxCity tutorials](index.md)

No account is needed: the default data sources are read from open repositories. Google Earth Engine is optional and only adds further sources ([guide](../guides/earth_engine.md)).

# VoxCity Data Sources Guide

Which sources exist for each layer, which need an account, what VoxCity picks automatically, and how to choose your own.

In [ ]:
# Install VoxCity (Google Colab or a fresh environment; skip if already installed)
%pip install -q voxcity

## What VoxCity picks automatically

`auto_select_data_sources` shows the choice for any location without downloading anything. `get_voxcity` uses the same rules; when Earth Engine is not initialized it swaps Earth Engine sources for login-free ones.

In [ ]:
from voxcity.generator import auto_select_data_sources

locations = {
    "Tokyo, Japan":   [(139.760, 35.680), (139.760, 35.684), (139.765, 35.684), (139.765, 35.680)],
    "New York, USA":  [(-74.015, 40.705), (-74.015, 40.709), (-74.010, 40.709), (-74.010, 40.705)],
    "London, UK":     [(-0.130, 51.500), (-0.130, 51.504), (-0.124, 51.504), (-0.124, 51.500)],
    "Lagos, Nigeria": [(3.380, 6.450), (3.380, 6.454), (3.384, 6.454), (3.384, 6.450)],
}
for name, vertices in locations.items():
    print(name)
    for layer, source in auto_select_data_sources(vertices).items():
        print(f"   {layer:32s} {source}")

## Sources by layer

Pass the name in the first column to `get_voxcity`. **EE** = needs Google Earth Engine.

### Buildings (`building_source`, `building_complementary_source`)

| Name | Coverage | Account | Notes |
|------|----------|---------|-------|
| `OpenStreetMap` | Worldwide | None | Default base source; heights where mapped |
| `Overture` | Worldwide | None | OSM plus other curated sources |
| `Microsoft Building Footprints` | N. America, Europe, Australia | None | Default complement there |
| `Global Building Atlas` | Worldwide | None | LOD1 footprints with heights |
| `EUBUCCO v0.1` | EU + Switzerland | None | Footprints with heights |
| `Open Building 2.5D Temporal` | Africa, Latin America, S & SE Asia | EE | Height raster; use as complement |
| `England 1m DSM - DTM` | England | EE | Heights from LiDAR, as complement |
| `Netherlands 0.5m DSM - DTM` | Netherlands | EE | Heights from LiDAR, as complement |
| `Local file` | Any | None | A GeoPackage given as `building_path` |

The complementary source fills heights (and, with `complement_polygon=True`, footprints) missing from the base source. `building_complement_height` sets the height for buildings still without one.

### Land cover (`land_cover_source`)

| Name | Coverage | Resolution | Account |
|------|----------|------------|---------|
| `OpenStreetMap` | Worldwide | Vector | None |
| `ESA WorldCover` | Worldwide | 10 m | None |
| `OpenEarthMapJapan` | Japan | ~1 m | None |
| `Urbanwatch` | 22 US cities | 1 m | EE |
| `ESRI 10m Annual Land Cover` | Worldwide | 10 m | EE |
| `Dynamic World V1` | Worldwide | 10 m | EE |

### Canopy height (`canopy_height_source`)

| Name | Coverage | Resolution | Account |
|------|----------|------------|---------|
| `High Resolution 1m Global Canopy Height Maps v2` | Worldwide | ~1.2 m | None (default) |
| `High Resolution 1m Global Canopy Height Maps` | Worldwide | 1 m | EE |
| `ETH Global Sentinel-2 10m Canopy Height (2020)` | Worldwide | 10 m | EE |
| `OpenStreetMap` | Worldwide | Mapped trees | None |
| `Static` | Anywhere | Land cover | None (fixed `static_tree_height`, default 10 m) |

### Terrain (`dem_source`)

| Name | Coverage | Resolution | Account |
|------|----------|------------|---------|
| `FABDEM` | Worldwide | 30 m | None |
| `COPERNICUS` | Worldwide | 30 m | None (surface model: includes buildings and trees) |
| `USGS 3DEP 10m` | USA | ~10 m | None |
| `GSI DEM Japan` | Japan | 5 m / 10 m | None |
| `Flat` | Anywhere | — | None |
| `USGS 3DEP 1m` | USA | 1 m | EE |
| `England 1m DTM` | England | 1 m | EE |
| `DEM France 1m` / `DEM France 5m` | France | 1 m / 5 m | EE |
| `Netherlands 0.5m DTM` | Netherlands | 0.5 m | EE |
| `AUSTRALIA 5M DEM` | Australia | 5 m | EE |
| `DeltaDTM` | Coastal lowlands | 30 m | EE |
| `NASA` | 60°N–56°S | 30 m | EE |

## Choosing sources yourself

In [ ]:
from voxcity.generator import get_voxcity

rectangle_vertices = locations["Tokyo, Japan"]

city = get_voxcity(
    rectangle_vertices,
    meshsize=5,
    building_source="Overture",
    building_complementary_source="OpenStreetMap",
    land_cover_source="ESA WorldCover",
    canopy_height_source="High Resolution 1m Global Canopy Height Maps v2",
    dem_source="GSI DEM Japan",
    output_dir="output/data_sources",
)
print(city.voxels.classes.shape)

### Useful options

| Option | Effect |
|--------|--------|
| `min_canopy_height` | Ignore trees lower than this (m) |
| `trunk_height_ratio` | Trunk height / tree height (default 0.59) |
| `static_tree_height` | Tree height for `canopy_height_source="Static"` (default 10 m) |
| `dem_interpolation` | Smooth the DEM when `meshsize` is finer than the DEM |
| `building_complement_height` | Height for buildings with no height in any source |
| `complement_polygon` | Also add footprints found only in the complementary source |

## Land cover classes

Every land cover source is converted to the same 14 VoxCity classes.

In [ ]:
from voxcity.utils.lc import get_land_cover_classes

for i, name in enumerate(dict.fromkeys(get_land_cover_classes("OpenStreetMap").values()), start=1):
    print(i, name)

## Earth Engine (optional)

Needed only for the sources marked EE above.

```python
import ee
ee.Authenticate()                               # on Colab: !earthengine authenticate --auth_mode=notebook
ee.Initialize(project="your-project-id")
```

Next: [demo_basic](demo_basic.ipynb) for the full workflow.